# Fase de Preparación de Datos — Pharma Sales Forecast

Este notebook implementa el pipeline de transformaciones para el dataset de Pharma Sales (`03_train_tablon_eda.pkl`) siguiendo el diseño acordado y las instrucciones de **A_04_PreparadorDatos.md**.

## Objetivos:
1. Cargar el tablón procedente del EDA.
2. Aplicar transformaciones en cascada usando `scikit-learn`:
   - **Fase 1**: Extraer y derivar variables de fecha (`day`, `dayofweek`, `weekofyear`, `is_weekend`).
   - **Fase 2**: Aplicar One-Hot Encoding a `weekday_name` y `granularity` con `drop='first'`.
   - **Fase 3**: Omitir reescalado (StandardScaler/MinMaxScaler) dado que se priorizan modelos basados en árboles (invariantes de escala).
   - **Fase 3.5**: Generar retardos (`t-1`, `t-2`) y medias móviles (`rolling_4`) agrupados por granularidad.
   - **Fase 4**: Unir las features, aplicar validaciones y guardar el tablón final listo para modelado.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

### 1. Carga del Dataframe

In [ ]:
pkl_path = '../02_datos/03_Entrenamiento/03_train_tablon_eda.pkl'
df = pd.read_pickle(pkl_path)
print(f"Dataset original cargado. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
df.head()

### FASE 1: Derivación de características numéricas temporales
Extraemos componentes del campo `date` y conservamos variables del año y del mes, además de la hora.

In [ ]:
df_fase1 = pd.DataFrame(index=df.index)
df_fase1['year'] = df['year'].astype(int)
df_fase1['month'] = df['month'].astype(int)

# Hora (contiene NaN para filas agregadas)
df_fase1['hour'] = pd.to_numeric(df['hour'], errors='coerce')

# Componentes desde date
date_series = pd.to_datetime(df['date'])
df_fase1['day'] = date_series.dt.day
df_fase1['dayofweek'] = date_series.dt.dayofweek
df_fase1['weekofyear'] = date_series.dt.isocalendar().week.astype(int)
df_fase1['is_weekend'] = date_series.dt.dayofweek.isin([5, 6]).astype(int)

print("Variables temporales derivadas:")
print(df_fase1.head())

### FASE 2: Codificación de variables categóricas (One-Hot Encoding)
Codificamos `weekday_name` y `granularity` utilizando `OneHotEncoder` de scikit-learn con `drop='first'` para evitar multicolinealidad.

In [ ]:
ohe = OneHotEncoder(drop='first', sparse_output=False)
categorical_cols = ['weekday_name', 'granularity']
ohe_array = ohe.fit_transform(df[categorical_cols])

# Nombres de columnas resultantes
ohe_cols = []
for i, col in enumerate(categorical_cols):
    cats = ohe.categories_[i][1:] # Omitimos el primero
    ohe_cols.extend([f"{col}_{cat}" for cat in cats])

df_fase2_binarias = pd.DataFrame(ohe_array, columns=ohe_cols, index=df.index)
print(f"Dummies generados ({len(ohe_cols)} columnas):")
print(df_fase2_binarias.head())

### FASE 3: Escalado selectivo
Dado que se entrenarán modelos basados en árboles, no se requiere aplicar escalado (como StandardScaler o MinMaxScaler) sobre las columnas continuas. Se omiten estos escaladores para mantener la interpretabilidad natural de los datos.

### FASE 3.5: Generación de Retardos (Lags) y Medias Móviles por Granularidad
Añadimos retardos de 1 y 2 periodos (`t-1` y `t-2`) y medias móviles de 4 periodos sobre los 8 targets. Estos cálculos se realizan agrupando por la columna original `granularity` y ordenando por `date` cronológicamente para evitar fugas de información temporales.

In [ ]:
df_lags = pd.DataFrame(index=df.index)
target_cols = ['m01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06']

for target in target_cols:
    df_temp = pd.DataFrame({
        'granularity': df['granularity'],
        'date': pd.to_datetime(df['date']),
        'val': df[target]
    }, index=df.index)
    
    df_temp = df_temp.sort_values(['granularity', 'date'])
    
    lag_1 = df_temp.groupby('granularity')['val'].shift(1)
    lag_2 = df_temp.groupby('granularity')['val'].shift(2)
    roll_4 = lag_1.groupby(df_temp['granularity']).rolling(window=4, min_periods=1).mean().reset_index(level=0, drop=True)
    
    df_lags[f'{target}_lag_1'] = lag_1
    df_lags[f'{target}_lag_2'] = lag_2
    df_lags[f'{target}_roll_mean_4'] = roll_4

print(f"Características de lag/rolling creadas: {df_lags.shape[1]} columnas.")
df_lags.head()

### FASE 4: Unión final y validaciones críticas

In [ ]:
target_cols = ['m01ab', 'm01ae', 'n02ba', 'n02be', 'n05b', 'n05c', 'r03', 'r06']
df_targets = df[target_cols].copy()

# Concatenar horizontalmente incluyendo las características temporales de lag/rolling
df_final = pd.concat([df_targets, df_fase1, df_fase2_binarias, df_lags], axis=1)

print("--- VALIDACIONES CRÍTICAS ---")
# 1. Conservación de filas
assert df_final.shape[0] == df.shape[0], f"Error en filas: {df_final.shape[0]} vs {df.shape[0]}"
print(f"✓ Validación 1 pasada: Filas conservadas ({df_final.shape[0]}) ")

# 2. Targets presentes
for col in target_cols:
    assert col in df_final.columns, f"Target {col} faltante"
print("✓ Validación 2 pasada: Targets presentes")

# 3. Sin columnas intermedias/excluidas
excluded = ['datum', 'year_month', 'date', 'weekday_name', 'granularity']
for col in excluded:
    assert col not in df_final.columns, f"Columna excluida {col} presente en df final"
print("✓ Validación 3 pasada: Columnas excluidas eliminadas")

# 4. Sin duplicados de nombres
assert len(df_final.columns) == len(set(df_final.columns)), "Nombres duplicados"
print("✓ Validación 4 pasada: Sin nombres de columna duplicados")

# 5. Sin correlación perfecta entre dummies (OHE con drop='first' correcto)
corr_matrix = df_final[ohe_cols].corr().abs()
perfect_corr = False
for i in range(len(ohe_cols)):
    for j in range(i+1, len(ohe_cols)):
        if corr_matrix.iloc[i, j] > 0.99:
            print(f"⚠️ Advertencia: Dummies correlacionados perfectamente: {ohe_cols[i]} y {ohe_cols[j]}")
            perfect_corr = True
if not perfect_corr:
    print("✓ Validación 5 pasada: Sin dummies correlacionados perfectamente")

### 5. Guardado del Dataframe Final

In [ ]:
pkl_out_path = '../02_datos/03_Entrenamiento/04_train_tablon_transformado.pkl'
os.makedirs(os.path.dirname(pkl_out_path), exist_ok=True)
df_final.to_pickle(pkl_out_path)
print(f"Dataframe final listo para modelización guardado en: {pkl_out_path}")

print("\n--- ESTRUCTURA DEL DATAFRAME TRANSFORMADO ---")
df_final.info()